In [1]:
import pandas as pd
from eu_cartel_innov.preproc import bulk_preproc
from eu_cartel_innov.matching import fuzzy_match
import numpy as np

# import and clean data

In [2]:
# orbis
orbis_data = pd.read_excel('../data/raw/orbis/orbis_base_15.12.25.xlsx', sheet_name='Risultati', dtype=str)
# regpat data
regpat_data = pd.read_csv('../data/raw/regpat/202505_PCT_App_reg.txt', sep='|', quotechar='"')
regpat_ipc = pd.read_csv('../data/raw/regpat/202505_PCT_IPC.txt', sep='|', quotechar='"')

In [3]:
orbis_data = (
    orbis_data
    .rename(columns={
    'Ragione socialeCaratteri latini': 'firm_name',
    'Ragione sociale precedente\nLatin Alphabet': 'prev_name',
    'Codice ISO paese': 'ctry_code',
    'Codice NACE Rev. 2, core code (4 cifre)': 'nace_code',
    'Data di costituzione': 'incorp_date',
    'Numero BvD ID': 'bvd_id'
    })
    .drop(columns=['Unnamed: 0'])
)

orbis_data['firm_name'] = orbis_data['firm_name'].ffill()
orbis_data['firm_id'] = orbis_data['firm_name'].factorize()[0]
orbis_data

,firm_name,prev_name,ctry_code,nace_code,incorp_date,bvd_id,firm_id
0,SAUDI ARABIAN OIL COMPANY SAUDI JOINT STOCK CO...,ARABIAN AMERICAN OIL COMPANY,SA,0610,32460,SA30947GS,0
1,CHINA PETROLEUM & CHEMICAL CORPORATION,NaN,CN,1920,36581,CN30086PC,1
2,PETROCHINA COMPANY LIMITED,NaN,CN,1920,36469,CN30081PC,2
3,VOLKSWAGEN AG,GESELLSCHAFT ZUR VORBEREITUNG DES DEUTSCHEN VO...,DE,2910,13663,DE2070000543,3
4,EXXON MOBIL CORPORATION,EXXON CORP,US,1920,05/08/1882,US135409005,4
...,...,...,...,...,...,...,...
8152,GIGASET AG,AKTIENGESELLSCHAFT BAD SALZSCHLIRF,NaN,NaN,NaN,NaN,8130
8153,COMPANIA SUD AMERICANA DE VAPORES S.A.,NaN,CL,5020,1872,CL901600007,8131
8154,DEUTSCHE LUFTHANSA AKTIENGESELLSCHAFT,NaN,DE,5110,9503,DE5190000974,8132
8155,CONTITECH DEUTSCHLAND GMBH,CONTITECH AG,DE,7010,33155,DE2190202355,8133


In [4]:
def extract_year(value):
    if pd.isna(value):
        return np.nan
    
    try: # number values (year only)
        num = float(value)
        if 1000 <= num <= 2100:
            return int(num)
        else: # excel serial dates
            date = pd.Timestamp("1899-12-30") + pd.to_timedelta(num, unit="D")
            return date.year
    except:
        pass
    
    try: # string dates
        date = pd.to_datetime(value, errors='coerce', dayfirst=True)
        if pd.notna(date):
            return date.year
    except:
        pass
    
    return np.nan

orbis_data['incorp_year'] = orbis_data['incorp_date'].apply(extract_year)

In [5]:
orbis_data = bulk_preproc(orbis_data, 'firm_name', keep_spaces=False)
orbis_data = bulk_preproc(orbis_data, 'prev_name', keep_spaces=False)
orbis_data.to_csv('../data/processed/clean_orbis_base.csv', index=False)

preprocessing 8,134 unique values out of 8,157 total rows...
created column: 'firm_name_preproc'
preprocessing 3,394 unique values out of 8,157 total rows...
created column: 'prev_name_preproc'


In [ ]:
regpat_data = regpat_data.merge(regpat_ipc, on=['pct_nbr'], how='left')
regpat_data = bulk_preproc(regpat_data, 'app_name', keep_spaces=False)
regpat_data.to_csv('../data/processed/pct_clean.csv', index=False)

# matching

## exact matching 

In [ ]:
regpat_data_unique = regpat_data[['app_name_preproc', 'ctry_code']].drop_duplicates(subset=['app_name_preproc', 'ctry_code'])

In [ ]:
# exact match on current firm names
exact_current = pd.merge(
    orbis_data[['firm_name_preproc', 'ctry_code', 'firm_id']],
    regpat_data_unique[['app_name_preproc', 'ctry_code']],
    left_on=['firm_name_preproc', 'ctry_code'],
    right_on=['app_name_preproc', 'ctry_code'],
    how='inner'
)
exact_current = exact_current.rename(columns={'firm_name_preproc': 'firm_preproc'})
exact_current['match_type'] = 'exact_current'

In [ ]:
# exact match on prev firm names
exact_prev = pd.merge(
    orbis_data[['prev_name_preproc', 'ctry_code', 'firm_id']],
    regpat_data_unique[['app_name_preproc', 'ctry_code']],
    left_on=['prev_name_preproc', 'ctry_code'],
    right_on=['app_name_preproc', 'ctry_code'],
    how='inner'
)
exact_prev = exact_prev.rename(columns={'prev_name_preproc': 'firm_preproc'})
exact_prev['match_type'] = 'exact_prev'

In [ ]:
exact_matches = pd.concat([exact_current, exact_prev], ignore_index=True)
print(exact_matches[['firm_id', 'firm_preproc', 'app_name_preproc', 'match_type']].head())

In [ ]:
# summary
num_current_matches = exact_current['firm_preproc'].nunique()
print(f"exact matches on current names: {num_current_matches}")
num_prev_matches = exact_prev['firm_preproc'].nunique()
print(f"exact matches on previous names: {num_prev_matches}")
num_exact_matches = exact_matches['firm_id'].nunique()
num_tot_orbis = orbis_data['firm_name_preproc'].nunique()
print(f"tot exact matches: {num_exact_matches}/{num_tot_orbis}")

## fuzzy matching

In [ ]:
# keep unmatched  firms
unmatched_current = orbis_data[~orbis_data['firm_name_preproc'].isin(exact_matches['firm_preproc'].unique())]
unmatched_prev = orbis_data[~orbis_data['prev_name_preproc'].isin(exact_matches['firm_preproc'].unique())]

In [ ]:
# fuzzy matching on current name
fuzzy_current = fuzzy_match(unmatched_current, regpat_data_unique, 'firm_name_preproc', 'app_name_preproc', min_score=90, match_type='fuzzy_current', ctry_col='ctry_code')

In [ ]:
fuzzy_current = pd.merge(
    fuzzy_current, 
    orbis_data[['firm_name_preproc', 'firm_id']], 
    on='firm_name_preproc', 
    how='left'
)
fuzzy_current = fuzzy_current.rename(columns={'firm_name_preproc': 'firm_preproc'})

In [ ]:
# fuzzy matching on previous name
fuzzy_prev = fuzzy_match(unmatched_prev, regpat_data_unique, 'prev_name_preproc', 'app_name_preproc', min_score=85, match_type='fuzzy_prev', ctry_col='ctry_code')

In [ ]:
fuzzy_prev = pd.merge(
    fuzzy_prev, 
    orbis_data[['prev_name_preproc', 'firm_id']], 
    left_on='prev_name_preproc', 
    right_on='prev_name_preproc', 
    how='left'
)
fuzzy_prev = fuzzy_prev.rename(columns={'prev_name_preproc': 'firm_preproc'})

In [ ]:
fuzzy_matches = pd.concat([fuzzy_current, fuzzy_prev], ignore_index=True)
print(fuzzy_matches[['firm_id', 'firm_preproc', 'app_name_preproc', 'match_type']].head())

## tot matches

In [ ]:
matches = pd.concat([exact_matches, fuzzy_matches], ignore_index=True)
matches.to_csv('../data/interim/firm_matches.csv', index=False)

In [ ]:
# summary
num_exact_matches = exact_matches['firm_id'].nunique()
num_fuzzy_matches = fuzzy_matches['firm_id'].nunique()
num_unmatched = orbis_data[~orbis_data['firm_id'].isin(matches['firm_id'])]['firm_id'].nunique()
num_tot_orbis = orbis_data['firm_name_preproc'].nunique()
num_tot_matches = matches['firm_id'].nunique()

match_rate = num_tot_matches / num_tot_orbis * 100

print(f"total matches: {num_exact_matches} exact and {num_fuzzy_matches} fuzzy")
print(f"unmatched Orbis firms: {num_unmatched}")
print(f"total firms found: {num_tot_matches} / {num_tot_orbis} ({match_rate:.2f}%)")